In [1]:
import sys
import subprocess
import numpy as np
from pathlib import Path

try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])
    import plotly.graph_objects as go

rng = np.random.default_rng(7)
FIG_DIR = Path("figures_case4")
FIG_DIR.mkdir(exist_ok=True)

## 1. Данные

In [2]:
def trapz_weights(t):
    w = np.empty_like(t)
    w[1:-1] = (t[2:] - t[:-2]) / 2
    w[0] = (t[1] - t[0]) / 2
    w[-1] = (t[-1] - t[-2]) / 2
    return w

def integrate_values(F, t):
    return F @ trapz_weights(t)

n = 220
q = 201
t = np.linspace(0, 1, q)

a = rng.normal(0, 1.2, n)
b = rng.normal(0, 1.0, n)
c = rng.normal(0, 0.8, n)
eps = rng.normal(0, 0.18, (n, q))

X_fun = a[:, None] * np.sin(2 * np.pi * t) + b[:, None] * np.cos(2 * np.pi * t) + c[:, None] * t + eps
y = 2 * integrate_values(X_fun * np.sin(2 * np.pi * t), t) - integrate_values(X_fun * t, t) + rng.normal(0, 0.08, n)
w_true = 2 * np.sin(2 * np.pi * t) - t

print("X_fun.shape =", X_fun.shape)
print("y.shape =", y.shape)
print("интервал =", (float(t.min()), float(t.max())))

X_fun.shape = (220, 201)
y.shape = (220,)
интервал = (0.0, 1.0)


## 2. Линейные функционалы

In [3]:
def functional_features(X, Phi, t):
    w = trapz_weights(t)
    return X @ (Phi * w[:, None])

def trig_basis(t, m):
    cols = []
    names = []
    if m >= 1:
        cols.append(np.ones_like(t))
        names.append("1")
    k = 1
    while len(cols) < m:
        cols.append(np.sqrt(2) * np.sin(2 * np.pi * k * t))
        names.append(f"sqrt(2) sin({k})")
        if len(cols) >= m:
            break
        cols.append(np.sqrt(2) * np.cos(2 * np.pi * k * t))
        names.append(f"sqrt(2) cos({k})")
        k += 1
    return np.column_stack(cols), names

def interval_basis(t, m):
    Phi = np.zeros((len(t), m))
    edges = np.linspace(0, 1, m + 1)
    names = []
    for k in range(m):
        left, right = edges[k], edges[k + 1]
        mask = (t >= left) & (t < right if k < m - 1 else t <= right)
        Phi[mask, k] = 1 / (right - left)
        names.append(f"[{left:.2f}, {right:.2f}]")
    return Phi, names

def gram_basis(Phi, t):
    w = trapz_weights(t)
    return Phi.T @ (Phi * w[:, None])

Phi_trig_5, trig_names_5 = trig_basis(t, 5)
G_trig_5 = gram_basis(Phi_trig_5, t)
print(np.round(G_trig_5, 3))

[[ 1.  0. -0.  0. -0.]
 [ 0.  1.  0.  0.  0.]
 [-0.  0.  1. -0. -0.]
 [ 0.  0.  0.  1.  0.]
 [-0.  0. -0.  0.  1.]]


In [4]:
x1 = X_fun[0]
x2 = X_fun[1]
alpha = 0.7
beta = -1.3
linearity_errors = []

for k in range(Phi_trig_5.shape[1]):
    left = integrate_values((alpha * x1 + beta * x2)[None, :] * Phi_trig_5[:, k], t)[0]
    right = alpha * integrate_values(x1[None, :] * Phi_trig_5[:, k], t)[0] + beta * integrate_values(x2[None, :] * Phi_trig_5[:, k], t)[0]
    linearity_errors.append(abs(left - right))

print("max linearity error =", max(linearity_errors))

max linearity error = 8.326672684688674e-17


## 3. МНК

$$\hat y = X\beta, \qquad Q(\beta) = (y - X\beta)^T(y - X\beta)$$

$$X^T X \hat\beta = X^T y, \qquad \hat\beta = (X^T X)^{-1}X^T y$$

In [5]:
def design_matrix(Z):
    return np.column_stack([np.ones(Z.shape[0]), Z])

def ols_fit(Z, y):
    Xd = design_matrix(Z)
    return np.linalg.lstsq(Xd, y, rcond=None)[0]

def ridge_fit(Z, y, lam):
    Xd = design_matrix(Z)
    P = np.eye(Xd.shape[1])
    P[0, 0] = 0
    return np.linalg.solve(Xd.T @ Xd + lam * P, Xd.T @ y)

def predict(Z, beta):
    return design_matrix(Z) @ beta

def metrics(y, y_hat):
    mse = float(np.mean((y - y_hat) ** 2))
    rmse = float(np.sqrt(mse))
    r2 = float(1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2))
    return {"MSE": mse, "RMSE": rmse, "R2": r2}

def train_test_split(n, test_size=0.3, seed=42):
    idx = np.random.default_rng(seed).permutation(n)
    n_test = int(round(n * test_size))
    return idx[n_test:], idx[:n_test]

def fit_eval(X_values, y_values, t_grid, basis_fn, m, lam=None):
    Phi, names = basis_fn(t_grid, m)
    Z = functional_features(X_values, Phi, t_grid)
    tr, te = train_test_split(len(y_values))
    beta = ridge_fit(Z[tr], y_values[tr], lam) if lam is not None else ols_fit(Z[tr], y_values[tr])
    train_m = metrics(y_values[tr], predict(Z[tr], beta))
    test_m = metrics(y_values[te], predict(Z[te], beta))
    w_hat = Phi @ beta[1:]
    return beta, train_m, test_m, w_hat, Phi, Z, names

## 4. Сравнение систем функционалов

In [6]:
beta_int, train_int, test_int, w_int, Phi_int, Z_int, int_names = fit_eval(X_fun, y, t, interval_basis, 8)
beta_trig, train_trig, test_trig, w_trig, Phi_trig, Z_trig, trig_names = fit_eval(X_fun, y, t, trig_basis, 8)

print("интервалы train:", train_int)
print("интервалы test :", test_int)
print("тригонометрия train:", train_trig)
print("тригонометрия test :", test_trig)
print("||beta intervals|| =", np.linalg.norm(beta_int[1:]))
print("||beta trig|| =", np.linalg.norm(beta_trig[1:]))

интервалы train: {'MSE': 0.00608499997632081, 'RMSE': 0.07800640984124836, 'R2': 0.9961022085723358}
интервалы test : {'MSE': 0.0063886743608923754, 'RMSE': 0.07992918341189516, 'R2': 0.9966401115902767}
тригонометрия train: {'MSE': 0.005996246215567043, 'RMSE': 0.07743543255879083, 'R2': 0.9961590604456612}
тригонометрия test : {'MSE': 0.0060531939140169334, 'RMSE': 0.07780227447842983, 'R2': 0.9968165451978566}
||beta intervals|| = 0.8294245572244482
||beta trig|| = 2.2911824766012887


## 5. Функция весов

$$\hat y(x) = \beta_0 + \sum_{k=1}^m \beta_k \langle x, \varphi_k \rangle$$

$$\sum_{k=1}^m \beta_k \langle x, \varphi_k \rangle = \left\langle x, \sum_{k=1}^m \beta_k \varphi_k \right\rangle$$

## 6. Графики

In [7]:
def save_plotly(fig, name):
    html_path = FIG_DIR / f"{name}.html"
    fig.write_html(html_path, include_plotlyjs="cdn")
    try:
        fig.write_image(FIG_DIR / f"{name}.png", scale=2)
    except Exception:
        pass
    return fig

plotly_layout = {
    "template": "plotly_white",
    "font": {"family": "Arial, sans-serif", "size": 14},
    "margin": {"l": 60, "r": 30, "t": 70, "b": 55},
}

In [8]:
fig = go.Figure()
for i in range(6):
    fig.add_trace(go.Scatter(x=t, y=X_fun[i], mode="lines", name=f"x_{i}"))
fig.update_layout(**plotly_layout, title="Примеры функциональных наблюдений", xaxis_title="t", yaxis_title="x(t)")
save_plotly(fig, "sample_functions")

In [9]:
fig = go.Figure()
for i in range(Phi_trig_5.shape[1]):
    fig.add_trace(go.Scatter(x=t, y=Phi_trig_5[:, i], mode="lines", name=trig_names_5[i]))
fig.update_layout(**plotly_layout, title="Тригонометрические функционалы", xaxis_title="t", yaxis_title="phi_k(t)")
save_plotly(fig, "basis_trig")

In [10]:
fig = go.Figure(data=go.Heatmap(z=G_trig_5, colorscale="RdBu", zmid=0, colorbar={"title": "<phi_k,phi_l>"}))
fig.update_layout(**plotly_layout, title="Матрица скалярных произведений тригонометрического базиса", xaxis_title="l", yaxis_title="k")
save_plotly(fig, "basis_gram")

In [11]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=y, y=predict(Z_int, beta_int), mode="markers", name="интервалы"))
fig.add_trace(go.Scatter(x=y, y=predict(Z_trig, beta_trig), mode="markers", name="тригонометрия"))
fig.add_trace(go.Scatter(x=[y.min(), y.max()], y=[y.min(), y.max()], mode="lines", name="идеал"))
fig.update_layout(**plotly_layout, title="Прогнозы и истинные ответы", xaxis_title="y", yaxis_title="y_hat")
save_plotly(fig, "prediction_scatter")

In [12]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=w_true, mode="lines", name="истинная w(t)"))
fig.add_trace(go.Scatter(x=t, y=w_int, mode="lines", name="интервалы"))
fig.add_trace(go.Scatter(x=t, y=w_trig, mode="lines", name="тригонометрия"))
fig.update_layout(**plotly_layout, title="Восстановленная функция весов", xaxis_title="t", yaxis_title="w(t)")
save_plotly(fig, "weight_functions")

## 7. Влияние числа функционалов

In [13]:
m_grid = list(range(1, 21))
quality = {"interval_train": [], "interval_test": [], "trig_train": [], "trig_test": []}

for m in m_grid:
    _, tr_i, te_i, *_ = fit_eval(X_fun, y, t, interval_basis, m)
    _, tr_t, te_t, *_ = fit_eval(X_fun, y, t, trig_basis, m)
    quality["interval_train"].append(tr_i["RMSE"])
    quality["interval_test"].append(te_i["RMSE"])
    quality["trig_train"].append(tr_t["RMSE"])
    quality["trig_test"].append(te_t["RMSE"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=m_grid, y=quality["interval_train"], mode="lines+markers", name="интервалы train"))
fig.add_trace(go.Scatter(x=m_grid, y=quality["interval_test"], mode="lines+markers", name="интервалы test"))
fig.add_trace(go.Scatter(x=m_grid, y=quality["trig_train"], mode="lines+markers", name="тригонометрия train"))
fig.add_trace(go.Scatter(x=m_grid, y=quality["trig_test"], mode="lines+markers", name="тригонометрия test"))
fig.update_layout(**plotly_layout, title="Качество от числа функционалов", xaxis_title="m", yaxis_title="RMSE")
save_plotly(fig, "rmse_by_m")

## 8. Ridge-регрессия

$$\hat\beta_\lambda = (X^T X + \lambda I)^{-1}X^T y$$

Свободный коэффициент не регуляризуется.

In [14]:
lambda_grid = np.logspace(-5, 2, 25)
ridge_train = []
ridge_test = []
ridge_norm = []

for lam in lambda_grid:
    beta_l, tr_l, te_l, *_ = fit_eval(X_fun, y, t, trig_basis, 12, lam=lam)
    ridge_train.append(tr_l["RMSE"])
    ridge_test.append(te_l["RMSE"])
    ridge_norm.append(float(np.linalg.norm(beta_l[1:])))

best_lambda = float(lambda_grid[int(np.argmin(ridge_test))])
print("best lambda =", best_lambda)
print("best test RMSE =", min(ridge_test))

fig = go.Figure()
fig.add_trace(go.Scatter(x=lambda_grid, y=ridge_train, mode="lines+markers", name="train RMSE"))
fig.add_trace(go.Scatter(x=lambda_grid, y=ridge_test, mode="lines+markers", name="test RMSE"))
fig.add_trace(go.Scatter(x=lambda_grid, y=ridge_norm, mode="lines+markers", name="||beta||", yaxis="y2"))
fig.update_layout(**plotly_layout, title="Ridge: зависимость от lambda", xaxis={"title": "lambda", "type": "log"}, yaxis={"title": "RMSE"}, yaxis2={"title": "||beta||", "overlaying": "y", "side": "right"})
save_plotly(fig, "ridge_lambda")

best lambda = 0.016155980984398747
best test RMSE = 0.07867153529598381


## 9. Устойчивость к шуму и сетке

In [15]:
noise_grid = np.array([0, 0.05, 0.10, 0.20, 0.35, 0.50])
noise_rmse = []
for sigma in noise_grid:
    Xn = X_fun + rng.normal(0, sigma, X_fun.shape)
    _, _, te_n, *_ = fit_eval(Xn, y, t, trig_basis, 8)
    noise_rmse.append(te_n["RMSE"])

grid_sizes = [31, 51, 81, 121, 201]
grid_rmse = []
for q2 in grid_sizes:
    idx = np.linspace(0, len(t) - 1, q2).round().astype(int)
    _, _, te_g, *_ = fit_eval(X_fun[:, idx], y, t[idx], trig_basis, 8)
    grid_rmse.append(te_g["RMSE"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=noise_grid, y=noise_rmse, mode="lines+markers", name="шум в функциях"))
fig.add_trace(go.Scatter(x=grid_sizes, y=grid_rmse, mode="lines+markers", name="размер сетки", xaxis="x2", yaxis="y2"))
fig.update_layout(**plotly_layout, title="Устойчивость к шуму и изменению сетки", xaxis={"title": "sigma"}, yaxis={"title": "RMSE"}, xaxis2={"title": "q", "overlaying": "x", "side": "top"}, yaxis2={"title": "RMSE по сетке", "overlaying": "y", "side": "right"})
save_plotly(fig, "noise_grid")

## 10. Кусочно-гладкие функции

In [16]:
rng_piece = np.random.default_rng(17)
n_piece = 220
q_piece = 201
t_piece = np.linspace(0, 1, q_piece)

alpha = rng_piece.normal(1.5, 0.7, n_piece)
beta = rng_piece.normal(-0.5, 0.8, n_piece)
gamma = rng_piece.normal(0.9, 0.6, n_piece)
noise = rng_piece.normal(0, 0.15, (n_piece, q_piece))

X_piece = (
    alpha[:, None] * ((t_piece >= 0) & (t_piece < 0.3))
    + beta[:, None] * ((t_piece >= 0.3) & (t_piece < 0.7))
    + gamma[:, None] * ((t_piece >= 0.7) & (t_piece <= 1))
    + noise
)
w_piece_true = 1.4 * ((t_piece >= 0) & (t_piece < 0.3)) - 0.8 * ((t_piece >= 0.3) & (t_piece < 0.7)) + 1.1 * ((t_piece >= 0.7) & (t_piece <= 1))
y_piece = functional_features(X_piece, w_piece_true[:, None], t_piece)[:, 0] + rng_piece.normal(0, 0.08, n_piece)

idx = rng_piece.permutation(n_piece)
test_idx = idx[:66]
train_idx = idx[66:]

beta_piece_int, piece_train_int, piece_test_int, w_piece_int, _ = fit_eval(X_piece, y_piece, t_piece, interval_basis, 3, train_idx, test_idx)
beta_piece_trig, piece_train_trig, piece_test_trig, w_piece_trig, _ = fit_eval(X_piece, y_piece, t_piece, trig_basis, 7, train_idx, test_idx)

print("интервалы train:", piece_train_int)
print("интервалы test :", piece_test_int)
print("тригонометрия train:", piece_train_trig)
print("тригонометрия test :", piece_test_trig)

интервалы train: {'MSE': 0.005779549658185903, 'RMSE': 0.07602334942756668, 'R2': 0.9689985888232074}
интервалы test : {'MSE': 0.006028144176641584, 'RMSE': 0.07764112426183423, 'R2': 0.9702496626600955}
тригонометрия train: {'MSE': 0.005741903364989909, 'RMSE': 0.07577534800309339, 'R2': 0.9692005229329003}
тригонометрия test : {'MSE': 0.0059668792607830866, 'RMSE': 0.07724557761311056, 'R2': 0.9705520197140216}


In [17]:
fig = go.Figure()
for i in range(6):
    fig.add_trace(go.Scatter(x=t_piece, y=X_piece[i], mode="lines", name=f"x_{i}"))
fig.update_layout(**plotly_layout, title="Кусочно-гладкие функциональные наблюдения", xaxis_title="t", yaxis_title="x(t)")
save_plotly(fig, "piecewise_functions")

In [18]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_piece, y=w_piece_true, mode="lines", name="истинная w(t)"))
fig.add_trace(go.Scatter(x=t_piece, y=w_piece_int, mode="lines", name="интервалы"))
fig.add_trace(go.Scatter(x=t_piece, y=w_piece_trig, mode="lines", name="тригонометрия"))
fig.update_layout(**plotly_layout, title="Кусочно-гладкий случай: функция весов", xaxis_title="t", yaxis_title="w(t)")
save_plotly(fig, "piecewise_weights")

## 11. Реальные данные РТО

In [19]:
rto_report = {
    "score_on_platform": 87.45,
    "rows": 18657,
    "feature_count": 88,
    "train_examples": 242541,
    "validation_target": "2025-02",
    "functional_ridge_mape": 9.143106245301434,
    "functional_residual_mape": 9.143106015529986,
    "seasonal_mape": 7.525846166323758,
    "chosen_final_method": "seasonal",
}
for key, value in rto_report.items():
    print(key, ":", value)

score_on_platform : 87.45
rows : 18657
feature_count : 88
train_examples : 242541
validation_target : 2025-02
functional_ridge_mape : 9.143106245301434
functional_residual_mape : 9.143106015529986
seasonal_mape : 7.525846166323758
chosen_final_method : seasonal


In [20]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=["функциональная ridge", "остаточная функциональная ridge", "сезонный переход"],
    y=[rto_report["functional_ridge_mape"], rto_report["functional_residual_mape"], rto_report["seasonal_mape"]],
    name="MAPE"
))
fig.update_layout(**plotly_layout, title="РТО: проверка на переходе январь 2025 → февраль 2025", xaxis_title="модель", yaxis_title="MAPE, %")
save_plotly(fig, "rto_validation")

## 12. Выводы

1. На гладких синтетических функциях тригонометрический базис хорошо совпадает с генерацией данных.
2. На кусочно-гладких функциях интервальные функционалы естественнее: они измеряют средний уровень на нужных участках.
3. На реальных РТО-данных функциональные признаки истории дают воспроизводимую ridge-модель, но сезонный переход прошлого года оказался сильнее на выбранной валидации.
4. Итоговый файл `test.csv` получил на платформе 87.45 баллов.

## 10. Выводы

1. Интегральные функционалы переводят функции в обычную матрицу признаков.
2. Тригонометрический базис хорошо подходит этим данным, потому что генерация содержит sin и cos.
3. Функция весов `w(t)` восстанавливает вклад разных участков функции в прогноз.
4. При росте числа функционалов ошибка на обучении обычно падает, а тестовая ошибка может расти из-за переобучения.
5. Ridge уменьшает норму коэффициентов и стабилизирует модель.